# MVO4 Swan 11A: REPLICATION_GUIDE.md implementation

This revised notebook uses the shared `mvo_algo.py` pipeline. Run from the `replication` directory. The uploaded notebook is preserved separately. The guide controls all dates, swans, and model choices. No live downloads are used.

In [1]:
import os, sys
from pathlib import Path
for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[name] = "1"
HERE = Path.cwd()
if not (HERE / "mvo_algo.py").exists():
    raise RuntimeError("Run this notebook from the replication directory")
sys.path.insert(0, str(HERE))
import pandas as pd
import matplotlib.pyplot as plt
from mvo_algo import Replication, EVENTS, versions, write_outputs
model = Replication(cache=HERE / ".cache/dbi")
versions()

{'python': '3.12.14',
 'numpy': '2.2.3',
 'pandas': '2.3.3',
 'scikit-learn': '1.7.2',
 'scipy': '1.16.3',
 'PyPortfolioOpt': '1.6.0',
 'cvxpy': '1.9.2'}

## 1–3. Historical inputs, plain-average crash shape, and inclusive window

In [2]:
EVENT = 'Swan 11A'
K, Seeds, MinAppear, w1 = 7, 3, 3, 0.17
ev = EVENTS[EVENT]
print(ev)
print("Eligible universe:", len(model.training_beta(EVENT)))
print("t1, t2, dd:", model.curve_params(EVENT))
print("start, end, extended:", model.training_window(EVENT))

Event(name='Swan 11A', peak='2025-02-19', bottom='2025-04-08', train_swans=('Swan6', 'Swan7', 'Swan8', 'Swan9', 'Swan10D'), optdata='revised_swan11a_sp100_optdata4.xlsx')
Eligible universe: 94
t1, t2, dd: (79, 198, 0.2564334269853163)
start, end, extended: (Timestamp('2023-12-30 00:00:00'), Timestamp('2025-02-14 00:00:00'), False)


## 4. Rank all 1,000 seeds, vote, retain the top 22 with at least MinAppear votes

In [3]:
ranking = model.dbi_ranking(EVENT, K)
print("Best seeds:", ranking[:Seeds])
selected, votes = model.preselect(EVENT, K, Seeds, MinAppear)
display(votes.to_frame())
display(selected)
assert len(selected) >= 5, "This combination has no_preselect status"

Best seeds: [(633, 0.9341472727499596), (988, 0.9341472727499597), (156, 0.9498484153395971)]


,votes
AMT,3
CL,3
DUK,3
EXC,3
KO,3
MCD,3
MDLZ,3
NEE,3
PEP,3
PG,3


,Swan6,Swan7,Swan8,Swan9,Swan10D,avg
Ticker,,,,,,
AMT,0.050751,0.583639,-0.336659,0.882155,1.271482,0.490274
CL,-0.045657,0.784583,0.569564,0.632697,0.619560,0.512150
DUK,-0.322793,0.233701,-0.356250,1.165454,0.534699,0.250962
EXC,0.263389,0.605246,-0.123438,1.254518,0.353268,0.470597
KO,0.325968,0.614677,-0.049526,1.161009,0.181359,0.446697
MCD,0.130072,0.483323,-0.202955,1.142392,0.376953,0.385957
MDLZ,0.344205,0.089729,0.386481,0.928976,0.451682,0.440215
NEE,0.415367,0.192033,-0.026635,1.062981,0.710514,0.470852
PEP,0.445938,0.591395,0.030199,0.863850,0.002890,0.386854


## 5–6. Inject at the front, then compute last-year drawdown and weekly covariance ×51

In [4]:
risk = model.risk_inputs(EVENT, selected)
print("Trading days:", risk["total_days"], "Cutoff:", risk["cutoff"])
pd.Series(risk["curve"], index=risk["raw_prices"].index).plot(title="Injected market drawdown")
plt.show()
display(pd.DataFrame({"fall": risk["fall"], "R_fwd_30_MA": risk["R_fwd"], "E_2": risk["E_2"]}))
display(risk["cov"])

Trading days: 282 Cutoff: 2024-02-15 00:00:00


,fall,R_fwd_30_MA,E_2
Ticker,,,
AMT,0.274839,0.251052,0.913452
CL,0.196982,0.117117,0.594559
DUK,0.121096,0.115212,0.951406
EXC,0.109922,0.116296,1.057985
KO,0.154914,0.149229,0.963299
MCD,0.156578,0.114047,0.728376
MDLZ,0.247737,0.224125,0.904688
NEE,0.207525,0.196400,0.946391
PEP,0.194471,0.153981,0.791795


Ticker,AMT,CL,DUK,EXC,KO,MCD,MDLZ,NEE,PEP,PG,SBUX,SO,VZ
Ticker,,,,,,,,,,,,,
AMT,0.089680,0.027631,0.030723,0.031899,0.020315,0.024225,0.030596,0.040915,0.022023,0.019989,0.005550,0.031618,0.022391
CL,0.027631,0.025798,0.013899,0.012433,0.017347,0.015384,0.020757,0.022249,0.016699,0.017404,0.002193,0.014908,0.013510
DUK,0.030723,0.013899,0.025083,0.022304,0.014361,0.015308,0.015354,0.025179,0.013879,0.012381,0.004581,0.022770,0.013449
EXC,0.031899,0.012433,0.022304,0.032519,0.013980,0.018396,0.017040,0.028098,0.013057,0.009947,0.012209,0.023881,0.018414
KO,0.020315,0.017347,0.014361,0.013980,0.023602,0.014987,0.018758,0.017469,0.015859,0.013057,0.013993,0.014892,0.014705
MCD,0.024225,0.015384,0.015308,0.018396,0.014987,0.039490,0.023459,0.018986,0.015469,0.010347,0.027637,0.017979,0.017163
MDLZ,0.030596,0.020757,0.015354,0.017040,0.018758,0.023459,0.033670,0.021846,0.020923,0.015844,0.021069,0.018886,0.016779
NEE,0.040915,0.022249,0.025179,0.028098,0.017469,0.018986,0.021846,0.063153,0.024046,0.015282,0.003201,0.029392,0.026291
PEP,0.022023,0.016699,0.013879,0.013057,0.015859,0.015469,0.020923,0.024046,0.030316,0.014475,0.011577,0.015622,0.011690


## 7–8. Max Sharpe at rf=0, 3% floor, then buy at peak and evaluate at bottom

In [5]:
result = model.run_one(EVENT, K, Seeds, MinAppear, w1, detail=True)
print("Status:", result["status"], "PR:", result["PR"])
assert result["status"] == "ok", result.get("error")
display(result["weights"].to_frame("weight"))
pd.DataFrame({"portfolio": result["evaluation"]["portfolio_values"], "benchmark": result["evaluation"]["benchmark_values"]}).plot(title=EVENT)
plt.show()

Status: ok PR: 0.9949758452273901


,weight
DUK,0.17000
EXC,0.17000
KO,0.17000
MDLZ,0.03651
PEP,0.11349
PG,0.17000
VZ,0.17000


## 9. Small sweep and optional exports

The command-line runner handles the complete 5,376-run grid. Below is a four-combination event sweep; failed optimization is retained as a status, with no alternate model.

In [6]:
small = model.run_sweep(dict(K=[7], Seeds=[3], MinAppear=[3], w1=[0.05, 0.11, 0.15, 0.17]), [EVENT])
display(small)
SAVE_OUTPUTS = False
if SAVE_OUTPUTS:
    output = HERE / "results" / EVENT.replace(" ", "_")
    write_outputs(small, output, [EVENT])
    model.export_checkpoints(output / "replication_checkpoints.xlsx", K=K, Seeds=Seeds, MinAppear=MinAppear, w1=w1)

Swan 11A, K=7: 4 runs completed


,event,K,Seeds,MinAppear,w1,status,PR,port_dd,market_dd,n_preselect,n_final,holdings,error,train_start,train_end
0,Swan 11A,7,3,3,0.05,opt_failed,NaN,NaN,NaN,13,0,{},infeasible: n_stocks * w1 < 1,2023-12-30,2025-02-14
1,Swan 11A,7,3,3,0.11,ok,0.991653,0.001660,0.198868,13,10,"{""CL"": 0.0768667275445797, ""DUK"": 0.1103830291...",None,2023-12-30,2025-02-14
2,Swan 11A,7,3,3,0.15,ok,0.997815,0.000434,0.198868,13,7,"{""DUK"": 0.15, ""EXC"": 0.15, ""KO"": 0.15, ""MDLZ"":...",None,2023-12-30,2025-02-14
3,Swan 11A,7,3,3,0.17,ok,0.994976,0.000999,0.198868,13,7,"{""DUK"": 0.17, ""EXC"": 0.17, ""KO"": 0.17, ""MDLZ"":...",None,2023-12-30,2025-02-14


Limitations: original checkpoint workbook is missing; fresh Yahoo downloads and alternative experimental objectives are outside this replication workflow. Swan9 analyst inputs have the guide's point-in-time caveat. See `IMPLEMENTATION_REPORT.md` for validation results.